# Файнтюнинг модели BART для суммаризации диалогов

В этом ноутбуке реализован end-to-end пайплайн дообучения (fine-tuning) предварительно обученной модели `ainize/bart-base-cnn` на наборе данных `knkarthick/dialogsum`.
Цель — научить модель генерировать краткие и точные выжимки (саммари) из длинных диалогов.

In [ ]:
# Установка необходимых библиотек (раскомментируйте при запуске в Colab/новом окружении)
# !pip install torch==1.13.1 torchdata==0.5.1 --quiet
# !pip install transformers==4.27.2 evaluate==0.4.0 rouge_score==0.1.2 datasets --quiet

In [ ]:
import time
import warnings
import evaluate
import torch
import pandas as pd
import numpy as np

from datasets import load_dataset
from transformers import (
    PreTrainedTokenizerFast,
    BartForConditionalGeneration,
    GenerationConfig,
    Trainer,
    TrainingArguments
)

# Отключаем предупреждения для чистоты вывода
warnings.filterwarnings("ignore")

# Если вы запускаете код в Google Colab и хотите сохранить веса на Диск,
# раскомментируйте строки ниже:
# from google.colab import drive
# drive.mount('/content/drive')

## 1. Загрузка данных и базовой модели
Используем датасет `dialogsum`, который содержит более 13 000 диалогов с написанными человеком аннотациями (саммари).

In [ ]:
# Загрузка набора данных
dataset_name = "knkarthick/dialogsum"
dataset = load_dataset(dataset_name)
print(dataset)

# Загрузка токенизатора и весов модели
model_name = "ainize/bart-base-cnn"
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

## 2. Предобработка (Токенизация)


In [ ]:
def tokenize_function(example):
    start_prompt = 'Summarize the following conversation.\n\n'
    end_prompt = '\n\nSummary: '

    # Формируем промпты для всего батча
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example["dialogue"]]

    max_length = 512

    # Токенизируем входные диалоги
    example['input_ids'] = tokenizer(
        prompt, padding="max_length", truncation=True, max_length=max_length, return_tensors="pt"
    ).input_ids

    # Токенизируем целевые саммари (labels)
    example['labels'] = tokenizer(
        example["summary"], padding="max_length", truncation=True, max_length=max_length, return_tensors="pt"
    ).input_ids

    return example

# Применяем токенизацию ко всему датасету
tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Удаляем текстовые колонки, оставляя только тензоры для PyTorch
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'topic', 'dialogue', 'summary'])

tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

print(f"Размер тренировочной выборки после фильтрации: {len(tokenized_datasets['train'])}")

## 3. Дообучение (Fine-tuning)

In [ ]:
output_dir = f'./dialogue-summary-training-{str(int(time.time()))}'

# Параметры гипертюнинга
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,              # Количество эпох
    per_device_train_batch_size=4,   # Размер батча на видеокарте
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",
    report_to="none"                 # Отключаем сторонние логгеры вроде wandb
)

# Создание объекта Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

# Запуск процесса обучения
trainer.train()

# Перемещение модели на GPU для ускорения генерации ответов (Inference)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

## 4. Оценка качества модели (Evaluation)

In [ ]:
# Загрузка метрики ROUGE
rouge = evaluate.load('rouge')

# Выбираем 10 примеров из тестовой выборки
dialogues = dataset['test'][0:10]['dialogue']
human_baseline_summaries = dataset['test'][0:10]['summary']

model_summaries =[]

# Генерация саммари
for dialogue in dialogues:
    prompt = f"Summarize the following conversation.\n\n{dialogue}\n\nSummary: "
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

    # Генерация выходных данных моделью
    model_outputs = model.generate(
        input_ids=input_ids,
        generation_config=GenerationConfig(max_new_tokens=200, early_stopping=True)
    )

    # Декодирование токенов обратно в текст
    model_text_output = tokenizer.decode(model_outputs[0], skip_special_tokens=True)
    model_summaries.append(model_text_output)

# Вычисляем ROUGE
model_results = rouge.compute(
    predictions=model_summaries,
    references=human_baseline_summaries,
    use_aggregator=True,
    use_stemmer=True,
)

print('РЕЗУЛЬТАТЫ МЕТРИКИ ROUGE ПОСЛЕ ФАЙНТЮНИНГА:')
for key, value in model_results.items():
    print(f"{key.upper()}: {value:.4f}")

## 5. Экспорт весов

In [ ]:
# Путь для сохранения.
# Для Google Colab: "/content/drive/MyDrive/bartbase"
# Для локального компьютера: "./saved_models/bart_finetuned"
model_path = "./saved_models/bart_finetuned"

# Сохранение
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

print(f"✅ Модель и токенизатор успешно сохранены в: {model_path}")